In [1]:
using Yao
using Optim
using LinearAlgebra
using StatsBase

# Problema de la mochila en digital

Habiendo implementado el problema de la mochila en computación analógica, hemos topado con límites cuando el número $N$ de activos pasa de las decenas y empieza a haber problemas para mapear el sistema en un grafo planar. Es por esto que vamos a dar el salto a la computación digital, ahora la correlación de los activos no será un problema, ya que se implementarán a nuestro problema mediante puertas $CZ$. Implementaremos el algoritmo QAOA (Quantum Approximate Optimization Algorithm), que evita el problema de la planaridad de los grafos.

## El Toy-Model

Repetiremos el mismo problema que en el cuaderno anterior, de todas formas, se plantea por cuestiones de contexto:

Tendremos cuatro T-bills con distintos valores de retorno y riesgo medio. La clave está en que ahora existe correlación entre los activos.

La situación es la siguiente, disponemos de los siguientes activos:
*  **Activo 1:** Retorno alto (8%), pero muy volátil (riesgo propio = 10).
*  **Activo 2:** Retorno bueno (7%), riesgo medio (5).
*  **Activo 3:** Retorno medio (5%), riesgo medio (6). Muy correlacionado con el **activo 1**, si se compran a la vez el riesgo conjunto se dispara.
*  **Activo 4:** Retorno bajo (3%), riesgo casi nulo (2).

Pasemos a analizar cómo son los vectores que representan esta situación. Definimos un vector de retornos $\mu = (8, 6, 5, 3)$ y una matriz de covarianza  $$\Sigma = \begin{pmatrix} 10 & 2 & 8 & 1 \\ 2 & 5 & 3 & 0 \\ 8 & 3 & 6 & 1 \\ 1 & 0 & 1 & 2 \end{pmatrix}$$
La última restricción es que solo tenemos presupuesto para comprar dos activos.

In [2]:
#Vector de retornos:
μ = [8.0, 7.0, 5.0 , 3.0]
N = length(μ)
#Matriz de correlaciones
sigma = [ 10.0 2.0 8.0 1.0;
           2.0 5.0 3.0 0.0;
           8.0 3.0 6.0 1.0;
           1.0 0.0 1.0 2.0];

## Transformación QUBO:

Para pasar a unas variables binarias $x_i \in \{0,1\}$ que representen si compramos o no un activo, definimos una función coste como sigue:
$$H = -\underbrace{\sum \mu_i x_i}_{\text{Retorno}} + \gamma \underbrace{\sum_{i,j} x_i \Sigma_{ij} x_j}_{\text{Riesgo}} + \lambda \underbrace{\left( \sum x_i - 2 \right)^2}_{\text{Restricción: Elegir 2}}$$

Expandiendo el término de restricción obtenemos:
$$\lambda (\sum x_i - 2)^2 = \lambda \left( (\sum x_i)^2 - 4\sum x_i + 4 \right)$$
Y aplicando la propiedad $x_i^2 = x_i$ podemos agrupar términos en dos tipos:
*  Términos lineales ($x_i$): forman la diagonal de una matriz Q.
*  Términos cuadráticos ($x_ix_j$): forman las interacciones de la matriz Q.

Pasamos entonces a calcular estos elementos. Para un número de activos $K$, la expansión de $\lambda (\sum{x_i-K})^2$ da unas fórmulas exactas para términos lineales y cuadráticos:
* Términos lineales (Diagonal $Q_{ii}$):
 $$Q_{ii} = -\mu_i + \gamma \Sigma_{ii} + \lambda (1 - 2K)$$
* Términos cuadráticos (Cruces $Q_{ij}$, para $i \neq j$):
  $$Q_{ij} = 2\gamma \Sigma_{ij} + 2\lambda$$

In [3]:
#Parámetros del gestor:
K = 2 #Número exacto de activos.
gamma_QUBO = 1.0 #Factor de aversión al riesgo.
lambda = 5.0 #Factor de penalización.

N_activos = length(μ)

#Construimos la matriz Q:
Q = zeros(Float64, N_activos, N_activos)

for i in 1:N_activos
    for j in 1:N_activos
        if i == j
            #Diagonal: términos lineales
            Q[i, i] = -μ[i] + gamma_QUBO * sigma[i, i] + lambda * (1 - 2*K)
        elseif j > i
            #Fuera de la diagonal: términos de cruce
            Q[i, j] =  gamma_QUBO * sigma[i, j] + lambda
            Q[j, i] = Q[i, j]
        end
    end
end
#Finalmente, redimensionamos la matriz Q para que sus entradas entren entre [-1,1]
escala = maximum(abs.(Q))
Q = Q ./ escala;

## Implementación del QAOA

### Hamiltoniano de coste
Una vez disponemos de nuestra matriz QUBO, la función a minimizar es la siguiente:
$$E(x) = \sum_{i} Q_{ii} x_i + 2 \sum_{i<j} Q_{ij} x_i x_j$$
(Se ignoran las constantes $\gamma$ y $\lambda$ porque no afectan a la optimización de los estados).

Para pasar al ordenador cuántico, debemos transformar las variables clásicas binarias $x_i$ a operadores cuánticos (matrices). Hacemos uso de las **Matrices de Pauli**, en concreto de la $Z$. La relación matemática para mapear bits $\{0,1\}$ a espines $\{1,-1\}$ es:
$$x_i = \frac{I - Z_i}{2}$$

Tras sustituir y agrupar términos, la función de coste clásica se transforma en el **Hamiltoniano de Coste ($H_C$):**
$$H_C = \sum_{i} h_i Z_i + \sum_{i<j} J_{ij} Z_i Z_j$$

#### Implementación del circuito
Para simular la evolución cuántica del estado aplicamos el operador unitario $U_C(\gamma) = e^{-i \gamma H_C}$.
* El término lineal $e^{-i \gamma h_i Z_i}$ se implementa aplicando una **puerta de rotación $R_z$** en el qubit $i$
* El término cuadrático $e^{-i \gamma J_{ij} Z_i Z_j}$ que representa la covarianza financiera se implementa mediante la secuencia de puertas **$CNOT$ - $R_z$ - $CNOT$** entre los qubits $i$ y $j$. Gracias al Atom Shuttling de Quera, esto se puede hacer sin importar si $i$ y $j$ son vecinos.

### Hamiltoniano de mezcla

Necesitamos ahora un operador que introduzca interferencia y permita al sistema "moverse" entre diferentes soluciones. Esto es el **Hamiltoniano de mezcla ($H_M$)**, basado en la matriz $X$ de Pauli:
$$H_M = \sum_{i=1}^{N} X_i$$
Aplicar el operador $X$ provoca un flip en el bit. Además, como los hamiltonianos no conmutan ($[H_C, H_M] \neq 0$), está garantizado que el sistema explora todo el Espacio de Hilbert.

#### Implementación del circuito

Aplicamos el operador unitario $U_M(\beta) = e^{-i \beta H_M}$, que en el circuito es una **puerta de rotación $R_x$** aplicada a todos los qubits.

### Preparación del estado

El algortimo debe comenzar en un estado donde todas las carteras posibles ($2^N$ estados) tengan la misma probabilidad de ser medidas. Este es el estado fundamental del Hamiltoniano de mezcla. Esto se consigue aplicando una **puerta de Hadamard ($H$)** a todos los qubits inicializados en $\ket{0}$:
$$|\psi_0\rangle = |+\rangle^{\otimes N} = \frac{1}{\sqrt{2^N}} \sum_{x=0}^{2^N-1} |x\rangle$$

Con la teoría asentada, pasemos a implementar en código este algoritmo:


In [4]:
#Construcción del circuito digital:
function capa_QAOA(N, gamma, beta, Q)
    capa = chain(N)
    h = zeros(N) 
    J = zeros(N,N)
    # Hamiltoniano de Coste: Traducimos la matriz QUBO en los h_i y J_ij.
    for i in 1:N
        h[i]= -Q[i,i] / 2.0
        
        for j in 1:N
            if i != j
                h[i] -= Q[i,j] /2.0
            end
        end
    end
    for i in 1:N
        for j in (i+1):N
            J[i,j] = Q[i,j] / 2.0
        end
    end
    
    for i in 1:N
        # Términos lineales (Retornos indiviauales de los activos)
        # Queremos aplicar exp(-i * γ * h_i * Z_i)
        # La puerta Rz(θ) aplica exp(-i * θ/2 * Z_i) -> θ = 2 * γ * h_i
        theta_lin = 2 * gamma * h[i]
        push!(capa, put(N, i => Rz(theta_lin))) #Aplicamos la rotación de ángulo theta_lin al qubit i

        # Términos cuadráticos (Riesgos cruzados)
        for j in (i+1):N
            if abs(Q[i,j]) > 0.001 #Si hay covarianza entre activos
                #Aplicamos la secuencia (CNOT - Rz - CNOT)
                # Queremos aplicar exp(-i * γ * J_ij * Z_i Z_j)
                # CNOT-Rz(θ)-CNOT aplica exp(-i * θ/2 * Z_i Z_j) -> θ = 2 * γ * J_ij
                theta_qua = 2 * gamma * J[i,j]
                push!(capa, cnot(N, i, j))
                push!(capa, put(N, j => Rz(theta_qua)))
                push!(capa, cnot(N, i, j))
            end
        end
    end
    # Hamiltoniano de mezcla. Forzamos al sistema a explorar otras carteras
    for i in 1:N
        # Queremos aplicar exp(-i * β * X_i)
        # Rx(θ) aplica exp(-i * θ/2 * X_i) -> θ = 2 * β
        push!(capa, put(N, i => Rx(2 * beta)))
    end
    return capa
end;

Pasamos ahora a implementar una función que prepare el circuito con $p$ capas, que se encargan de garantizar que el sistema alcance el estado óptimo de manera análoga a un barrido adiabático. La profundidad del circuito viene dado por el número de capas $p$. Teóricamente, si $p \to \infty$ se garantiza matemáticamente encontrar la solución óptima absolutaa. En la práctica de la era NIST, nos conformamos con $p \in [1, 5]$ para no introducir mucho ruido.

In [5]:
function circuito_QAOA(N, p, gammas, betas, Q)
    circuito = chain(N)
    #Comenzamos construyendo el estado inicial
    for i in 1:N
        push!(circuito, put(N, i => H)) #Aplicamos puertas Hadamard a todos los qubits
    end

    #Añadimos las capas repetidas p veces
    for it in 1:p
        push!(circuito, capa_QAOA(N, gammas[it], betas[it], Q))
    end
    return circuito
end;

## Optimización

Ya tenemos las funciones que determinan cómo evoluciona nuestro sistema. El resultado será un estado final $|\psi(\boldsymbol{\gamma}, \boldsymbol{\beta})\rangle$, del cuál obtendremos el valor esperado de la energía mediante:
$$E(\boldsymbol{\gamma}, \boldsymbol{\beta}) = \langle \psi(\boldsymbol{\gamma}, \boldsymbol{\beta}) | H_C | \psi(\boldsymbol{\gamma}, \boldsymbol{\beta}) \rangle$$

Pero hasta ahora no hemos definido cómo vamos a determinar los ángulos $\gamma$ y $\beta$. Para determinar los valores óptimos para estos parámetros vamos a realizar un estudio mediante gradiente descendente haciendo uso de la libreria  `Optim.jl`:

1. Partimos de unos ángulos aleatorios
2. Ejecutamos el circuito, colapsamos la función de onda final y devolvemos la energía $E$
3. Se calcula el gradiente (cómo hay que cambiar los ángulos para minimizar $E$)
4. Proponemos nuevos ángulos y empezamos en 1.

Repetimos el bucle hasta converger en los ángulos óptimos

In [6]:
p = 10 #Capas de profundidad

function perdida(parametros) #Parámetros es un vector con p valores para gamma y p valores para beta
    gammas = parametros[1:p]
    betas = parametros[p+1:end]

    #Ejecutamos el circuito
    reg = zero_state(N)
    circuito = circuito_QAOA(N, p, gammas, betas, Q)
    apply!(reg,circuito)
    
    #Medimos las probabilidades de cada estado
    probabilidades = statevec(reg) .|> abs2
    energia_esperada = 0.0
    
    #Medimos el valor esperado de la función coste
    for i in 0:(2^N-1)
        bits = digits(i, base=2, pad=N) #Convertimos estado a vector binario
        energia_qubo = dot(bits, Q * bits) 
        energia_esperada +=probabilidades[i+1] * energia_qubo
    end
    return energia_esperada
end

mejor_energia = Inf
mejores_parametros = zeros(2 * p)

for intento in 1:50

    gammas_init = sort(rand(p)) .* 2π
    betas_init = sort(rand(p), rev=true) 
    
    parametros_iniciales = vcat(gammas_init, betas_init)
    resultado_opt = optimize(perdida, parametros_iniciales, LBFGS(), Optim.Options(iterations=2500))
    energia_actual = Optim.minimum(resultado_opt)
    if energia_actual < mejor_energia
        mejor_energia = energia_actual 
        mejores_parametros = Optim.minimizer(resultado_opt)
    end
end

LoadError: ParseError:
[90m# Error @ [0;0m]8;;file://C:/Users/riosb/Programacion/TFG/QuEra/Ejemplos/RiskAnalysis/In[6]#34:92\[90mIn[6]:34:92[0;0m]8;;\
    parametros_iniciales = vcat(gammas_init, betas_init)
    resultado_opt = optimize(perdida, parametros_iniciales, LBFGS(), Optim.Options(iterati [48;2;120;70;70mons=2500[0;0m))
[90m#                                                                                          └──────┘ ── [0;0m[91mExpected `)`[0;0m

Una vez tenemos los mejores ángulos posibles, ejecutamos el circuito una última vez y realizamos las mediciones finales.

Gracias a la interferencia constructiva de los operadores unitarios aplicados durante las $p$ capas, los estados subóptimos se cancelan entre sí, mientras que la amplitud del estado óptimo se amplifica.

In [7]:
#Ejecutamos el circuito una última vez con los ángulos perfectos
reg_final = zero_state(N)
circuito_optimo = circuito_QAOA(N, p, mejores_parametros[1:p], mejores_parametros[p+1:end], Q)
apply!(reg_final, circuito_optimo)

#Medimos el estado final
resultados_finales = statevec(reg_final) .|> abs2

#Mostramos las tres carteras mejores
estados = [digits(i, base=2, pad=N) for i in 0:(2^N-1)]
carteras = [(estados[i], resultados_finales[i]) for i in 1:length(estados)]
sort!(carteras, by=x -> x[2], rev=true)
println("\n --- TOP 3 CARTERAS ---")
for i in 1:3
    activos = [j for j in 1:N if carteras[i][1][j] == 1]
    probabilidad = round(carteras[i][2] * 100, digits = 2)
    println("$i. Probabilidad: $probabilidad % -> Comprar Activos:", isempty(activos) ? "Ninguno" : join(activos, ", "))
end


 --- TOP 3 CARTERAS ---
1. Probabilidad: 6.25 % -> Comprar Activos:Ninguno
2. Probabilidad: 6.25 % -> Comprar Activos:1
3. Probabilidad: 6.25 % -> Comprar Activos:2


## Caso con N=10 y K = 6



In [8]:
# ── Datos del problema: Escenario Realista N=10, K=6 ──────────────────────────
N = 10
K = 6

# 1. Vector de retornos esperados (μ) en porcentaje
μ = [11.5, 10.2, 9.0, 8.5, 7.5, 6.0, 5.5, 4.0, 2.5, 1.8]

# 2. Construcción de Matriz de Covarianza (Sigma) mediante modelo de factores
# Las columnas representan: [Beta de Mercado, Exposición Tech, Activo Refugio]
cargas_factoriales = [
    2.5   2.0  -0.5;  # 1. Tech A (Alta volatilidad, alta correlación)
    2.2   1.8  -0.4;  # 2. Tech B
    2.0   1.5  -0.3;  # 3. Tech C
    1.8   0.0  -0.2;  # 4. Consumo Cíclico A (Sigue al mercado)
    1.6   0.0  -0.1;  # 5. Consumo Cíclico B
    1.0  -0.2   0.5;  # 6. Salud (Sector defensivo)
    0.9  -0.2   0.6;  # 7. Consumo Básico (Sector defensivo)
    0.6  -0.5   1.0;  # 8. Utilities (Baja volatilidad)
   -0.2  -0.5   1.5;  # 9. Bono Corporativo (Cobertura moderada)
   -0.5  -0.8   1.8   # 10. Bono del Tesoro (Cobertura fuerte frente a crisis)
]

# Matriz base (Cargas * Cargas transpuesta)
sigma = cargas_factoriales * cargas_factoriales'

# Añadimos el riesgo idiosincrático (riesgo propio e independiente de cada activo)
riesgo_propio = [2.0, 1.8, 1.5, 1.5, 1.2, 1.0, 0.8, 0.6, 0.3, 0.1]
for i in 1:N
    sigma[i, i] += riesgo_propio[i]
end

# Redondeamos para mantener la limpieza en la visualización
sigma = round.(sigma, digits=2);

#Parámetros del gestor:
gamma_QUBO = 1.0 #Factor de aversión al riesgo.
lambda = 10.0 #Factor de penalización.

N_activos = length(μ)

#Construimos la matriz Q:
Q = zeros(Float64, N_activos, N_activos)

for i in 1:N_activos
    for j in 1:N_activos
        if i == j
            #Diagonal: términos lineales
            Q[i, i] = -μ[i] + gamma_QUBO * sigma[i, i] + lambda * (1 - 2*K)
        elseif j > i
            #Fuera de la diagonal: términos de cruce
            Q[i, j] =  gamma_QUBO * sigma[i, j] + lambda
            Q[j, i] = Q[i, j]
        end
    end
end
#Finalmente, redimensionamos la matriz Q para que sus entradas entren entre [-1,1]
escala = maximum(abs.(Q))
Q = Q ./ escala;

In [ ]:
p = 12 #Capas de profundidad

energias_precalculadas = zeros(2^N)
for i in 0:(2^N-1)
    bits = digits(i, base=2, pad=N)
    energias_precalculadas[i+1] = dot(bits, Q * bits)
end

# 2. Redefinimos la función de pérdida para que sea ultrarrápida
function perdida(parametros)
    gammas = parametros[1:p]
    betas = parametros[p+1:end]

    # Ejecutamos el circuito
    reg = zero_state(N)
    circuito = circuito_QAOA(N, p, gammas, betas, Q)
    apply!(reg, circuito)
    
    # Medimos las probabilidades de cada estado
    probabilidades = statevec(reg) .|> abs2
    
    # Simplemente hacemos el producto punto entre las probabilidades y las energías precalculadas
    energia_esperada = dot(probabilidades, energias_precalculadas)
    
    return energia_esperada
end

mejor_energia = Inf
mejores_parametros = zeros(2 * p)

for intento in 1:50

    gammas_init = sort(rand(p)) .* 2π
    betas_init = sort(rand(p), rev=true) 
    
    parametros_iniciales = vcat(gammas_init, betas_init)
    resultado_opt = optimize(perdida, parametros_iniciales, LBFGS(), Optim.Options(iterations=2000))
    energia_actual = Optim.minimum(resultado_opt)
    if energia_actual < mejor_energia
        mejor_energia = energia_actual
        mejores_parametros = Optim.minimizer(resultado_opt)
    end
end

In [ ]:
#Ejecutamos el circuito una última vez con los ángulos perfectos
reg_final = zero_state(N)
circuito_optimo = circuito_QAOA(N, p, mejores_parametros[1:p], mejores_parametros[p+1:end], Q)
apply!(reg_final, circuito_optimo)

#Medimos el estado final
resultados_finales = statevec(reg_final) .|> abs2

#Mostramos las tres carteras mejores
estados = [digits(i, base=2, pad=N) for i in 0:(2^N-1)]
carteras = [(estados[i], resultados_finales[i]) for i in 1:length(estados)]
sort!(carteras, by=x -> x[2], rev=true)
println("\n --- TOP 3 CARTERAS ---")
for i in 1:3
    activos = [j for j in 1:N if carteras[i][1][j] == 1]
    probabilidad = round(carteras[i][2] * 100, digits = 2)
    println("$i. Probabilidad: $probabilidad % -> Comprar Activos:", isempty(activos) ? "Ninguno" : join(activos, ", "))
end